# Session 3 — Residual bounds and effectivity

Download the notebook using the toolbar and run all cells in order before modifying the marked settings.
The defaults provide a working baseline. Complete the tasks by changing the experiment and interpreting the results.
NumPy and Matplotlib are sufficient; there are no external data files.
Website results are generated at build time. The downloadable notebook is editable in Jupyter.
## Rebuild a small reduced model (10 minutes)

The supplied model and training parameters match session 2. Use rank two initially.
**This practical reports Euclidean state errors**, $G=I$, so both the dual residual norm and coercivity constant use that convention.
The basis can still be normalized in $hI$: Galerkin reduction depends on its span, not that normalization.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n = 64
h = 1.0 / (n + 1)
x = np.arange(1, n + 1) * h
K = (2*np.eye(n) - np.eye(n, k=1) - np.eye(n, k=-1)) / h**2
I = np.eye(n)
f = np.ones(n)


def full(mu):
    return np.linalg.solve(mu*K + I, f)


# Fixed training samples; geometric midpoints are distinct held-out samples.
mu_train = np.geomspace(0.1, 10.0, 12)
mu_test = np.sqrt(mu_train[:-1] * mu_train[1:])
S = np.column_stack([full(mu) for mu in mu_train])
U, sigma, Vt = np.linalg.svd(np.sqrt(h / len(mu_train)) * S, full_matrices=False)

r = 2  # Task: compare 1 and 3 after the baseline run.
Z = U[:, :r] / np.sqrt(h)
Kr, Mr, fr = Z.T @ K @ Z, Z.T @ Z, Z.T @ f
lambda_min_K = 4 / h**2 * np.sin(np.pi / (2*(n+1)))**2


def reduced(mu):
    a = np.linalg.solve(mu*Kr + Mr, fr)
    return a, Z @ a


def alpha_exact(mu):
    return 1 + mu * lambda_min_K


**Task 1.** Derive `alpha_exact` from the smallest eigenvalue of the Dirichlet difference matrix. Why is `1.0` also a valid lower bound for all parameters here?
## Compare actual error and two bounds (20 minutes)

The full solution is used only to evaluate the estimator in this experiment.
The certificate itself uses the residual and a stability lower bound.


In [ ]:
errors, bounds, weak_bounds, effectivities, output_errors = [], [], [], [], []
for mu in mu_test:
    a, ur = reduced(mu)
    rho = f - (mu*K + I) @ ur
    error = np.linalg.norm(full(mu) - ur)
    delta = np.linalg.norm(rho) / alpha_exact(mu)
    errors.append(error)
    bounds.append(delta)
    weak_bounds.append(np.linalg.norm(rho))  # alpha_LB = 1
    effectivities.append(delta / error if error > 1e-12 else np.nan)
    output_errors.append(abs(h * np.sum(full(mu) - ur)))
print(f"effectivity range={np.nanmin(effectivities):.4f} to {np.nanmax(effectivities):.4f}")
print(f"maximum state error={max(errors):.6e}")
print(f"maximum certified bound={max(bounds):.6e}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].loglog(mu_test, errors, "o-", label="Actual Euclidean state error")
axes[0].loglog(mu_test, bounds, "s--", label="Bound: exact coercivity")
axes[0].loglog(mu_test, weak_bounds, "^:", label="Bound: lower bound 1")
axes[0].set(xlabel="Parameter mu", ylabel="Euclidean error / bound")
axes[0].legend(fontsize=8)
axes[1].semilogx(mu_test, effectivities, "o-")
axes[1].axhline(1, color="black", linestyle="--", label="Coverage threshold")
axes[1].set(xlabel="Parameter mu", ylabel="Effectivity")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()


**Task 2.** Check coverage on all test parameters with an appropriate floating-point tolerance. Compare the sharpness of the two valid bounds. Does an effectivity larger than one mean the reduced model is inaccurate?
**Task 3.** Form an output bound using $\|\ell\|_2=h\sqrt n$. Compare it with `output_errors`. Explain the difference between certifying the discrete model and predicting a real component&#8217;s temperature accurately.
## Remove the full-order residual from the online query (20 minutes)

Precompute residual columns and two factorizations offline. The QR factor gives an alternative to evaluating a quadratic form with cancellation.
The full residual below is retained solely for validation.


In [ ]:
B = np.column_stack([f, K @ Z, Z])
residual_gram = B.T @ B
Qres, Tres = np.linalg.qr(B, mode="reduced")
mu_probe = 0.35
ap, urp = reduced(mu_probe)
c = np.r_[1.0, -mu_probe * ap, -ap]
direct_norm = np.linalg.norm(f - (mu_probe*K + I) @ urp)
gram_square = float(c @ residual_gram @ c)
qr_norm = np.linalg.norm(Tres @ c)
print(f"direct residual norm={direct_norm:.8e}")
print(f"Gram residual squared={gram_square:.8e}")
print(f"QR residual norm={qr_norm:.8e}")
print(f"full dimension={n}; residual-factor shape={Tres.shape}")


**Task 4.** Implement an online function returning the scalar output and QR-based bound using only `Kr`, `Mr`, `fr`, `Tres`, the reduced output vector and `alpha_exact`. It must neither call `full` nor form an $n$-vector.
**Task 5.** Repeat the residual comparison for ranks 1, 2, 3 and 4. Inspect `gram_square` before taking its square root. Explain why silently clipping every negative value to zero could hide a failed estimator. At extremely small residuals, report absolute discrepancies and a roundoff floor rather than claiming arbitrary relative accuracy.

## Discussion and submission checkpoint (10 minutes)

Keep one coverage plot, your output-bound check, and your online function with a list of its input sizes.
Show on paper why changing the norm to $G=hI$ changes both the dual residual norm and the coercivity constant.
Explain how maximizing a reliable estimator over a training set could select the next snapshot for a greedy reduced basis.
